# Candy Crush RL Training on Google Colab

## 步骤概览
1. 挂载Google Drive
2. 克隆GitHub仓库
3. 安装依赖
4. 编译C++扩展
5. 测试环境
6. 启动TensorBoard
7. 运行训练

In [ ]:
# 1. 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 设置工作目录
import os
WORK_DIR = '/content/drive/MyDrive/candy_crush_rl'
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

In [ ]:
# 2. 克隆GitHub仓库（如果尚未克隆）
import os

REPO_URL = "https://github.com/Pliaustjn/candy_crush_rl.git"
REPO_DIR = os.path.join(WORK_DIR, 'candy_crush_rl')

if not os.path.exists(REPO_DIR):
    !git clone -b master {REPO_URL} {REPO_DIR}
    print(f"Repository cloned to {REPO_DIR}")
else:
    %cd {REPO_DIR}
    !git pull origin master
    print("Repository updated")

%cd {REPO_DIR}

In [ ]:
# 3. 安装依赖
!pip install pybind11 numpy torch torchvision gym tensorboard

# 验证安装
import pybind11
import torch
import gym
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"Gym version: {gym.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. 编译C++扩展
import subprocess
import sys
import os

# 添加src目录到Python路径
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

print("Generating C++ source files...")
%run generate_clean_files.py

print("\nCompiling C++ extension...")
!python setup.py build_ext --inplace

# 验证编译结果
import candy_crush_cpp
print("\nTesting compiled module...")
env = candy_crush_cpp.CandyCrushEnv(max_steps=5, target_score=100, seed=42)
board = env.reset()
print(f"Environment created successfully!")
print(f"Board shape: {len(board)}x{len(board[0])}")
print(f"Steps: {env.get_steps()}, Score: {env.get_score()}")

# Test Gym wrapper - 修复导入问题
print("\nTesting Gym wrapper...")
from candy_crush_network import create_action_mapping
from gym_env_wrapper import CandyCrushGymEnv

gym_env = CandyCrushGymEnv(max_steps=5, target_score=100)
obs = gym_env.reset()
print(f"Gym environment created!")
print(f"Observation shape: {obs.shape}")
print(f"Action space: {gym_env.action_space}")
print(f"Observation space: {gym_env.observation_space}")

In [ ]:
# 5. 运行Gym演示脚本
print("Running Gym environment demo...")
%run colab_demo.py

In [ ]:
# 6. 设置TensorBoard
%load_ext tensorboard

# 在后台启动TensorBoard
%tensorboard --logdir training_logs_*/tensorboard --port 6006

In [ ]:
# 7. 运行训练（使用原始train_mcts_cpp.py的参数）
import sys
import os

# 添加当前目录和src目录到路径
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# 直接运行原始训练脚本
print("Starting training with original parameters...")
%run src/train_mcts_cpp.py

In [ ]:
# 8. 训练完成后，将模型保存到Google Drive
import shutil
from datetime import datetime

# 复制模型文件到Google Drive
MODEL_SAVE_DIR = os.path.join(WORK_DIR, 'saved_models')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 复制所有.pth文件
for file in os.listdir('.'):
    if file.endswith('.pth'):
        src = os.path.join('.', file)
        dst = os.path.join(MODEL_SAVE_DIR, f"{timestamp}_{file}")
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

# 复制训练日志
for file in os.listdir('.'):
    if file.endswith('.txt') and file.startswith('training_log'):
        src = os.path.join('.', file)
        dst = os.path.join(MODEL_SAVE_DIR, f"{timestamp}_{file}")
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

print(f"\nAll models and logs saved to {MODEL_SAVE_DIR}")

In [ ]:
# 9. 可选：GitHub自动提交（如果训练成功）
def git_commit_and_push(commit_message="Update training results"):
    """自动提交并推送训练结果到GitHub"""
    import subprocess
    
    try:
        # 配置Git（Colab环境）
        !git config --global user.email "pliaustjn@gmail.com"
        !git config --global user.name "Pliaustjn"
        
        # 添加新文件
        !git add *.pth training_logs_*/
        
        # 提交
        !git commit -m "{commit_message}"
        
        # 推送到master分支
        !git push origin master
        
        print("Successfully pushed to GitHub!")
    except Exception as e:
        print(f"Git push failed: {e}")
        print("You can manually push later.")

# 取消注释以启用自动推送
# git_commit_and_push()